# Nettoyage de la base d'apprentissage

In [1]:
import numpy as np
import pandas as pd
import sys
import os
import dill as pickle

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

In [2]:
# On charge la base d'apprentissage et celle des classements FIFA
df = pd.read_csv("../data_finale/base_apprentissage.csv")
df_fifa = pd.read_csv("../data/classement_fifa/fifa_ranking_fin_saison.csv", sep=",", encoding="utf-8-sig")

## Traitement des doublons

In [3]:
verifier_doublons_metier_et_techniques(df)

Recherche de doublons
 Parfait ! Aucune répétition technique (Même joueur, même saison, même club).

881 lignes correspondent à des doublons de mercato
(Même joueur, même saison, mais clubs différents -> Transferts de mi-saison)

Exemple de joueurs transférés concernés :
                    player  season     team
0   Ainsley Maitland-Niles    2021  Arsenal
14             Joe Willock    2021  Arsenal
16         Martin Ødegaard    2021  Arsenal
17             Mathew Ryan    2021  Arsenal
25          Sead Kolašinac    2021  Arsenal
26        Shkodran Mustafi    2021  Arsenal


{'doublons_techniques': np.int64(0), 'doublons_mercato': np.int64(881)}

In [4]:
df = fusionner_doublons_techniques(df)

Format initial de la base : (16815, 121)
Format après fusion intelligente des doublons : (16815, 121)


In [5]:
df = fusionner_et_recalculer_mercato(df)

Format avant fusion mercato : (16815, 121)
Format après fusion mercato : (15934, 121)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:284: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:284: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)


## Homogénéisation des formats

In [6]:
colonnes_dates = ["date_of_birth", "contract_expiration_date"]

# Application ciblée
df = nettoyer_age_et_dates(
    df, colonnes_dates=colonnes_dates
)

Traitement des colonnes de dates : ['date_of_birth', 'contract_expiration_date']
 -> Toutes les heures ont été remises à minuit.
 -> Colonne 'born' (année de naissance) extraite.
Calcul et nettoyage de la colonne 'age'...
 -> Attention : 0 âges manquants remplacés par la médiane (25 ans).
 -> Colonne 'age' convertie strictement en entiers (int).
Nettoyage de l'âge et des dates terminé.



## Traitement des variables avec beaucoup de valeurs manquantes

In [7]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 94.7% de valeurs manquantes
   • Performance_CS% : 92.8% de valeurs manquantes
   • Performance_Save% : 92.6% de valeurs manquantes
   • Performance_Saves : 92.5% de valeurs manquantes
   • Performance_GA : 92.5% de valeurs manquantes
   • Performance_GA90 : 92.5% de valeurs manquantes
   • Performance_SoTA : 92.5% de valeurs manquantes
   • Penalty Kicks_PKsv : 92.5% de valeurs manquantes
   • Penalty Kicks_PKm : 92.5% de valeurs manquantes
   • Penalty Kicks_PKA : 92.5% de valeurs manquantes
   • Performance_D : 92.5% de valeurs manquantes
   • Performance_CS : 92.5% de valeurs manquantes
   • Penalty Kicks_PKatt : 92.5% de valeurs manquantes
   • Performance_L : 92.5% de valeurs manquantes
   • Performance_W : 92.5% de valeurs manquantes
   • xg_buildup : 47.3% de valeurs manquantes
   • xg : 47.3% de va

In [8]:
# Application de la fonction
df = nettoyer_valeurs_manquantes_ciblees(df)

Début du traitement ciblé des valeurs manquantes...
 -> 1549 lignes supprimées car 'market_value_in_eur' était manquant.
 -> 25 colonnes de performance nettoyées (NaN -> 0).
 -> Propagation inter-saisons terminée pour 5 colonnes fixes.
Finitions terminées (Derniers NaN résiduels convertis en valeurs neutres).


In [9]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • xa : 46.7% de valeurs manquantes
   • xg : 46.7% de valeurs manquantes
   • xg_chain : 46.7% de valeurs manquantes
   • np_xg : 46.7% de valeurs manquantes
   • xg_buildup : 46.7% de valeurs manquantes
   • Subs_Mn/Sub : 14.0% de valeurs manquantes
   • contract_expiration_date : 9.1% de valeurs manquantes
   • Starts_Mn/Start : 6.8% de valeurs manquantes

Total : 8 colonnes dépassent le seuil de 1%.


## Encodage de variables

In [10]:
# Analyser la base
var_categorielles = lister_variables_categorielles(df)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • player (5100 modalités uniques)
   • team (137 modalités uniques)
   • league (5 modalités uniques)
   • nation (125 modalités uniques)
   • pos (10 modalités uniques)
   • join_key (5099 modalités uniques)
   • match_method (67 modalités uniques)
   • sub_position (13 modalités uniques)
   • position (5 modalités uniques)
   • foot (3 modalités uniques)
   • name (5100 modalités uniques)

Total : 11 variables catégorielles trouvées.


In [11]:
# Variables catégorielles à traiter
mes_variables = ["pos", "sub_position", "nation", "league", "foot"]

# Lancement de l'encodage
df = encoder_dataset_football(
    df=df,
    colonnes_categoriques=mes_variables,
    df_fifa_historique=df_fifa,
)

Format initial avant encodage : (14385, 121)
Encodage de 'nation' en 10 colonnes binaires Top FIFA (par saison)...
   • Variable 'confederation' encodée en colonnes binaires.
   • Les 10 colonnes classement_FIFA_X ont été injectées.
Profil Joueurs de champ détecté : Encodage Multi-Label de 'pos'.
Encodage One-Hot des colonnes : ['sub_position', 'league', 'foot']
Format final après encodage : (14385, 159)



## Jours contrat restants

In [12]:
df = calculer_jours_contrat_restants(df)

Début du calcul de la durée restante des contrats...
Contrats déjà expirés (valeur négative) : 2
Contrats manquants (NaN)                  : 1305

Statistiques descriptives de la variable calculée :
count    13080.000000
mean      1398.613838
std        736.065259
min        -30.000000
25%        731.000000
50%       1461.000000
75%       1826.000000
max       5113.000000
Calcul terminé avec succès.


### PROBLÈME : LES DATES DE FIN DE CONTRAT NE SONT PAS LES BONNES

**Constat :** Les informations de fin de contrat actuellement présentes dans notre dataset correspondent aux données **actuelles** extraites rétroactivement (par exemple, un contrat courant jusqu'au *30 juin 2035*). 

**Impact :** Cela fausse le calcul de la variable `contrat_jours_restants`.

## Suppression de colonnes en double

In [13]:
colonnes_redondantes = [
    "Starts_Starts", "Standard_PK", "Standard_PKatt", "Standard_Gls", "90s", "Playing Time_Min%",
    "Performance_SoTA", "Performance_G+A", "Team Success_+/-", "Team Success_+/-90", "Playing Time_Min", 
    "Penalty Kicks_PKatt", "born", "np_xg", "xg_chain", "Per 90 Minutes_G+A-PK", "Per 90 Minutes_G-PK",
    "join_key", "tm_join_key", "tm_join_key_full", "tm_id", "player_id", "dob_key", "tm_dob_key",
    "match_method", "name", "date_of_birth", "dob_year", "date", "season", "valuation_season_year",
    "contract_expiration_date", "Starts_Mn/Start", "Performance_W",
    "Performance_D", "Performance_L"
]

df = supprimer_colonnes_du_dataset(df, colonnes_redondantes)

30 colonne(s) supprimée(s) : ['Starts_Starts', 'Standard_PK', 'Standard_PKatt', 'Standard_Gls', '90s', 'Playing Time_Min%', 'Performance_SoTA', 'Performance_G+A', 'Team Success_+/-', 'Team Success_+/-90', 'Playing Time_Min', 'Penalty Kicks_PKatt', 'born', 'np_xg', 'xg_chain', 'Per 90 Minutes_G+A-PK', 'Per 90 Minutes_G-PK', 'join_key', 'tm_id', 'dob_key', 'match_method', 'name', 'date_of_birth', 'season', 'valuation_season_year', 'contract_expiration_date', 'Starts_Mn/Start', 'Performance_W', 'Performance_D', 'Performance_L']


## Sauvegarde de la pipeline de premier nettoyage

In [14]:
# Sauvegarde de la fonction dans le fichier .pkl
with open("../data_finale/pipelines/pipeline_nettoyage.pkl", "wb") as fichier:
    pickle.dump(executer_pipeline_nettoyage, fichier)

## Traitement des outliers et normalisations

In [15]:
dossier_sortie = r"..\data_finale"

cols_a_normaliser = [
    "market_value_in_eur",
    "Playing Time_MP",
    "Playing Time_Starts",
    "Starts_Compl",
    "Subs_Subs",
    "Subs_Mn/Sub",
    "Subs_unSub",
    "Performance_Gls",
    "Performance_Ast",
    "Performance_PK",
    "Performance_PKatt",
    "Performance_Saves",
    "Performance_Save%",
    "Performance_CS",
    "Performance_CS%",
    "Standard_Sh",
    "Standard_SoT",
    "Standard_SoT%",
    "Standard_G/Sh",
    "Standard_G/SoT",
    "Team Success_PPM",
    "xg",
    "xa",
    "xg_buildup",
]

In [16]:
df_train, df_val, df_test = executer_pipeline_preprocessing(
    df = df,
    dossier_sortie=dossier_sortie,
    cols_a_normaliser=cols_a_normaliser,
    methode_split="temporel",
    height_min=155,
    height_max=210,
)

2266 lignes de la saison 2025 ont été écartées du pipeline.
Split effectué -> Train: 7404 | Val: 2428 | Test: 2287
Outliers traités (Bornes: [155, 210] | Remplacement par la médiane du poste du Train).
Pipeline terminé ! Fichiers sauvegardés dans : ..\data_finale



## Sauvegarde de la pipeline de second nettoyage

In [17]:
# Sauvegarde de la fonction dans le fichier .pkl
with open("../data_finale/pipelines/pipeline_anti_data_leakage.pkl", "wb") as fichier:
    pickle.dump(executer_pipeline_preprocessing, fichier)

In [18]:
df

,player,team,nation,age,Playing Time_MP,Playing Time_Starts,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G-PK,...,sub_position_Second Striker,league_ENG-Premier League,league_ESP-La Liga,league_FRA-Ligue 1,league_GER-Bundesliga,league_ITA-Serie A,foot_both,foot_left,foot_right,contrat_jours_restants
0,Aaron Anselmino,Strasbourg,ARG,20,9,4,4.3,1,0,1,...,0,0,0,1,0,0,0,0,1,0.0
1,Aaron Connolly,Brighton,IRL,20,17,9,8.8,2,1,2,...,0,1,0,0,0,0,0,0,1,1096.0
2,Aaron Cresswell,West Ham United,ENG,31,36,36,35.2,0,8,0,...,0,1,0,0,0,0,0,1,0,1826.0
3,Aaron Cresswell,West Ham United,ENG,32,31,31,30.3,2,3,2,...,0,1,0,0,0,0,0,1,0,1461.0
4,Aaron Cresswell,West Ham United,ENG,33,28,24,24.8,0,1,0,...,0,1,0,0,0,0,0,1,0,1096.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14380,Šime Vrsaljko,Atlético Madrid,CRO,28,9,6,5.8,0,0,0,...,0,0,1,0,0,0,0,0,1,NaN
14381,Šime Vrsaljko,Atlético Madrid,CRO,29,21,10,10.2,1,2,1,...,0,0,1,0,0,0,0,0,1,NaN
14382,Ștefan Radu,Lazio,ROU,34,31,30,27.3,0,3,0,...,0,0,0,0,0,1,0,1,0,NaN
14383,Ștefan Radu,Lazio,ROU,35,10,6,6.2,0,0,0,...,0,0,0,0,0,1,0,1,0,NaN
